# Import Libraries

In [8]:
import pandas as pd
import os
import subprocess
from openpyxl import load_workbook
import matplotlib.pyplot as plt
import seaborn as sns

# Helper Functions

In [9]:
def loadExperimentPlan(path):
    return pd.read_excel(path)

In [10]:
def loadDatabase(path):
    return pd.read_excel(path, sheet_name="Julia")

In [11]:
def startProcessingExperiments(experimentPlan, flag):
    if flag is False:
        return
    for index, row in experimentPlan.iterrows():
        # Convert each row to a list
        currentExperiment = row.values.tolist()
        # Format template = [1, 3, 'Julia', 'CPU', 'Single-Core', 1, 'Gradient Descent', 10]
        if currentExperiment[2]   == "Julia":
            runJuliaEnvironment(currentExperiment)
        elif currentExperiment[2] == "Python":
            runPythonEnvironment(currentExperiment)
        elif currentExperiment[2] == "MATLAB":
            runMATLABEnvironment(currentExperiment)
        elif currentExperiment[2] == "Rust":
            runRustEnvironment(currentExperiment)

In [12]:
def runJuliaEnvironment(inputArgs):
    juliaScript = r"S:\Research Material\CSE\MU\Project\1Code\SuperScript\runJulia.py"
    # Convert all arguments to strings
    inputArgs = [str(arg) for arg in inputArgs]
    # Run the script with arguments and capture the output
    result = subprocess.run(['python', juliaScript] + inputArgs, capture_output=True, text=True)
    
    # Print the output of the script
    print("Output from runJulia.py:")
    print(result.stdout)
    
    # If there is any error, print it
    if result.stderr:
        print("Error from runJulia.py:")
        print(result.stderr)

In [13]:
def runPythonEnvironment(currentExperiment):
    pass

In [14]:
def runMATLABEnvironment(currentExperiment):
    pass

In [15]:
def runRustEnvironment(currentExperiment):
    pass

In [16]:
def processData(flag):
    if flag is False:
        return
    # Correct path joining and use raw string for the directory
    path = os.path.join(r"S:\Research Material\CSE\MU\Project\1Code\SuperScript", "processResultsAndPopulateDatabase.py")
    # Check if the script exists at the specified path
    if not os.path.exists(path):
        print(f"Error: The script file does not exist at {path}")
        return

    try:
        # Construct the command to run the Python script
        result = subprocess.run(['python', path], capture_output=True, text=True)

        # Check if the script ran successfully
        if result.returncode == 0:
            print(result.stdout)
        else:
            print("Python script encountered an error. Error details:")
            print(result.stderr)

    except Exception as e:
        print(f"An error occurred while running the Python script: {e}")

In [17]:
def visualize(databasePath, saveFigDir):
    # Load the data
    data = loadDatabase(databasePath)
    
    # Step 1: Drop the 'Unique ID' column
    data = data.drop(columns=['Unique ID'])
    
    # Step 2: Remove rows with N/A values
    data = data.dropna()
    
    # Step 3: Create the scatter plot subset (exclude 'R001')
    scatterData = data[data['Run ID'] != 'R001']
    
    # Step 4: Create the bar chart subset (only 'R001')
    r001Data = data[data['Run ID'] == 'R001']
    
    # Step 5: Group scatterData by 'Batch ID', 'Algorithm Type', 'Number Of Iterations', and calculate mean
    groupedScatterData = scatterData.groupby(['Batch ID', 'Algorithm Type', 'Number Of Iterations']).agg({'Total Script Run Time (s)': 'mean'}).reset_index()
    
    # Step 6: Create the Scatter and Bar directories if they don't exist
    scatterDir = os.path.join(saveFigDir, 'Scatter')
    barDir = os.path.join(saveFigDir, 'Bar')
    os.makedirs(scatterDir, exist_ok=True)
    os.makedirs(barDir, exist_ok=True)

    # Set the Seaborn style with white background
    sns.set_style('whitegrid')
    
    # Step 7: Plot scatter plot
    plt.figure(figsize=(10, 6))
    for algorithm in groupedScatterData['Algorithm Type'].unique():
        subset = groupedScatterData[groupedScatterData['Algorithm Type'] == algorithm]
        plt.scatter(subset['Number Of Iterations'], subset['Total Script Run Time (s)'], label=algorithm)
    plt.title('Mean Total Script Run Time by Number of Iterations (Excluding R001)', fontsize=12)
    plt.xlabel('Number Of Iterations', fontsize=12)
    plt.ylabel('Mean Total Script Run Time (s)', fontsize=12)
    plt.legend(fontsize=12)
    
    # Despine the plot (remove top and right spines)
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Apply tight layout to prevent overlap
    plt.tight_layout()

    # Save the scatter plot with 300 DPI and font size 12
    scatterPlotPath = os.path.join(scatterDir, 'scatter_plot.png')
    plt.savefig(scatterPlotPath, dpi=300)
    plt.close()

    # Step 8: Plot bar chart for R001 runs (x-axis is Number of Iterations)
    plt.figure(figsize=(10, 6))
    r001Grouped = r001Data.groupby(['Number Of Iterations']).agg({'Total Script Run Time (s)': 'mean'}).reset_index()
    plt.bar(r001Grouped['Number Of Iterations'], r001Grouped['Total Script Run Time (s)'])
    plt.title('Total Script Run Time for R001 by Number of Iterations', fontsize=12)
    plt.xlabel('Number Of Iterations', fontsize=12)
    plt.ylabel('Total Script Run Time (s)', fontsize=12)
    
    # Despine the bar chart (remove top and right spines)
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Apply tight layout to prevent overlap
    plt.tight_layout()

    # Save the bar chart with 300 DPI and font size 12
    barChartPath = os.path.join(barDir, 'bar_chart_r001.png')
    plt.savefig(barChartPath, dpi=300)
    plt.close()

    return scatterPlotPath, barChartPath

# Main Function

In [18]:
def main():
    # Location where the experimental setup plan is located.
    experimentSetup  = os.path.join(r"S:\Research Material\CSE\MU\Project\0ExperimentPlan\ExperimentalPlan.xlsx")
    # Get the experimental plan as a dataframe
    expPlanDataFrame = loadExperimentPlan(experimentSetup)
    # Process it now, row-by-row
    flag = True
    startProcessingExperiments(expPlanDataFrame, flag)
    processData(flag)
    # Save figures location
    visualPath   = os.path.join(r"S:\Research Material\CSE\MU\Project\3Visualizations")
    databasePath = os.path.join(r"S:\Research Material\CSE\MU\Project\2Database\Database.xlsx")
    visualize(databasePath, visualPath)

In [19]:
if __name__ == "__main__":
    main()

Output from runJulia.py:
Runs Per Batch: 5, Language: Julia, Compute Media: CPU, Concurrency Mode: Single-Core, Cores Used: 1, Algorithm Type: Gradient Descent, Number of Iterations: 1
Starting new batch Batch001 with command: julia S:\Research Material\CSE\MU\Project\1Code\AlgorithmAnalysis.jl\1CPU\src\startupCPU.jl 1 1 S:\Research Material\CSE\MU\Project\6Logs\Batch001 5
Batch Batch001 encountered an error.
Error details:   Activating project at `S:\Research Material\CSE\MU\Project\1Code\AlgorithmAnalysis.jl\1CPU`

Completed batch Batch001. Moving to the next batch...
All batches have been processed.

Output from runJulia.py:
Runs Per Batch: 5, Language: Julia, Compute Media: CPU, Concurrency Mode: Single-Core, Cores Used: 1, Algorithm Type: Gradient Descent, Number of Iterations: 2
Starting new batch Batch002 with command: julia S:\Research Material\CSE\MU\Project\1Code\AlgorithmAnalysis.jl\1CPU\src\startupCPU.jl 1 2 S:\Research Material\CSE\MU\Project\6Logs\Batch002 5
Batch Batch00